In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StringType, DoubleType, ArrayType
from pyspark.sql.functions import from_json, col, window, explode, to_timestamp, when, lit, collect_list, struct, to_json, expr

### Create Spark session

In [27]:
jars = os.environ.get('SPARK_JARS', '')

spark = SparkSession.builder \
    .config("spark.jars", jars) \
    .appName("BTC Z-score") \
    .master("local[*]") \
    .getOrCreate()

### Read data from `btc-price` topic

Read and parse the data from the `btc-price` topic

In [28]:
# Read data from btc-price topic
# btc-price publish its data from inside the container to outside using 9092 port
df_price = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "broker:9092") \
    .option("subscribe", "btc-price") \
    .load()
    
# get kafka value and convert it from binary to string format
df_string_price = df_price.selectExpr("CAST(value AS STRING) as json_value")

# cause the data is structured in json format, define a schema to parse it
schema_price = StructType() \
    .add("symbol", StringType()) \
    .add("price", StringType()) \
    .add("timestamp", StringType())
    
# Parse JSON
parsed_price_df = df_string_price.select(from_json(col("json_value"), schema_price).alias("data")).select("data.*")

# Cast timestamp format from String type to TimeStamp type
parsed_price_df = parsed_price_df.withColumn("timestamp", to_timestamp("timestamp", "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'")) \
                                .withWatermark("timestamp", "10 seconds")

For ease of debugging, run this cell to see the parsed data.

In [29]:
# Print to console (OPTIONAL) to check whether the data is read correctly or not
query_price = parsed_price_df.writeStream \
    .outputMode("update") \
    .format("console") \
    .start()

query_price.awaitTermination()

25/06/16 13:30:39 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-5cc97313-570f-4c39-bd36-87854a3ac533. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/06/16 13:30:39 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+------+-----+---------+
|symbol|price|timestamp|
+------+-----+---------+
+------+-----+---------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------+---------------+-------------------+
| symbol|          price|          timestamp|
+-------+---------------+-------------------+
|BTCUSDT|106711.35000000|2025-06-16 13:30:40|
+-------+---------------+-------------------+

-------------------------------------------
Batch: 2
-------------------------------------------
+-------+---------------+-------------------+
| symbol|          price|          timestamp|
+-------+---------------+-------------------+
|BTCUSDT|106711.34000000|2025-06-16 13:30:41|
+-------+---------------+-------------------+

-------------------------------------------
Batch: 3
-------------------------------------------
+-------+---------------+-------------------+
| s

-------------------------------------------
Batch: 12
-------------------------------------------
+-------+---------------+-------------------+
| symbol|          price|          timestamp|
+-------+---------------+-------------------+
|BTCUSDT|106736.66000000|2025-06-16 13:30:51|
+-------+---------------+-------------------+

-------------------------------------------
Batch: 13
-------------------------------------------
+-------+---------------+-------------------+
| symbol|          price|          timestamp|
+-------+---------------+-------------------+
|BTCUSDT|106736.66000000|2025-06-16 13:30:53|
+-------+---------------+-------------------+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/bitnami/python/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/python/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/python/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

-------------------------------------------
Batch: 14
-------------------------------------------
+-------+---------------+-------------------+
| symbol|          price|          timestamp|
+-------+---------------+-------------------+
|BTCUSDT|106744.19000000|2025-06-16 13:30:54|
+-------+---------------+-------------------+

-------------------------------------------
Batch: 15
-------------------------------------------
+-------+---------------+-------------------+
| symbol|          price|          timestamp|
+-------+---------------+-------------------+
|BTCUSDT|106763.43000000|2025-06-16 13:30:55|
+-------+---------------+-------------------+

-------------------------------------------
Batch: 16
-------------------------------------------
+-------+---------------+-------------------+
| symbol|          price|          timestamp|
+-------+---------------+-------------------+
|BTCUSDT|106763.43000000|2025-06-16 13:30:56|
+-------+---------------+-------------------+

-------------

In [31]:
# Stop the debug stream
query_price.stop()

### Read data from `btc-price-moving` topic

Read and parse the data from the `btc-price-moving` topic

In [32]:
# Read data from btc-price-moving topic
# btc-price-moving publish its data from inside the container to outside using 9092 port
df_moving = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "broker:9092") \
    .option("subscribe", "btc-price-moving") \
    .load()
    
# Get kafka value and convert it from binary to string format
df_string_moving = df_moving.selectExpr("CAST(value AS STRING) as json_value")

# Define schema for each sliding window
schema_window = StructType() \
    .add("window", StringType()) \
    .add("avg_price", DoubleType()) \
    .add("std_price", DoubleType())

# Define schema for each record which consists of array of different duration sliding windows defined above
schema_moving = StructType() \
    .add("timestamp", StringType()) \
    .add("symbol", StringType()) \
    .add("windows", ArrayType(schema_window))

# Parse JSON
parsed_moving_df = df_string_moving.select(from_json(col("json_value"), schema_moving).alias("data")) \
                                .select(to_timestamp(col("data.timestamp"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'").alias("timestamp"), 
                                        "data.symbol",
                                        explode("data.windows").alias("win"))  

Explode the sliding window in each record.

In [33]:
# Cause the record consists of array of different duration sliding windows, we have to explode them 
# to a single window with corresponding timestamp in the record
flat_moving_df = parsed_moving_df.select(
    col("timestamp").alias("moving_ts"),
    col("symbol").alias("moving_sym"),
    col("win.window").alias("window_type"),       # "30s", "1m", ...
    col("win.avg_price").alias("mean"),
    col("win.std_price").alias("std")
)

For ease of debugging, run this cell to see the parsed data.

In [34]:
# Print to console (OPTIONAL) to check whether the data is read correctly or not
query_moving = flat_moving_df.writeStream \
    .outputMode("update") \
    .format("console") \
    .start()

query_moving.awaitTermination()

25/06/16 13:31:21 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c3f3bb52-6932-4d57-a768-db65f1cbef45. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/06/16 13:31:21 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+----------+-----------+----+---+
|moving_ts|moving_sym|window_type|mean|std|
+---------+----------+-----------+----+---+
+---------+----------+-----------+----+---+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------------------+----------+-----------+----------+---------------+
|          moving_ts|moving_sym|window_type|      mean|            std|
+-------------------+----------+-----------+----------+---------------+
|2025-06-16 13:31:00|   BTCUSDT|        30s|106772.805|11.925405653473|
|2025-06-16 13:31:00|   BTCUSDT|         1m|106772.805|11.925405653473|
+-------------------+----------+-----------+----------+---------------+

-------------------------------------------
Batch: 2
-------------------------------------------
+-------------------+----------+-----------+------------------+--------------------+
|          mov

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/bitnami/python/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/python/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/python/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [35]:
# Stop the debugging stream
query_moving.stop()

### Join two topic by sliding window

Join records in topic `btc-price` and `btc-price-moving` by its correspond sliding window.

In [36]:
# List of sliding windows and its duration
windows = [("30s", "30 seconds"), 
           ("1m", "1 minute"), 
           ("5m", "5 minutes"), 
           ("15m", "15 minutes"), 
           ("30m", "30 minutes"), 
           ("1h", "1 hour")]

window_zscore = None
for label, duration in windows:
    price = parsed_price_df \
        .withColumn("window", window("timestamp", duration))

    moving = flat_moving_df \
        .filter(col("window_type") == label)

    joined = price.join(moving,
                    (col("symbol") == col("moving_sym")) &
                    (col("window.start") == col("moving_ts")),
                    "inner"
                   )

    # handle the case where standard deviation is equal to zero
    z_col = when(col("std") == 0, 0).otherwise((col("price") - col("mean")) / col("std"))
    
    result = joined.select(
        "timestamp", 
        "symbol", 
        lit(label).alias("window_type"),
        z_col.alias("zscore_price")
    )
        
    if window_zscore is None:
        window_zscore = result
    else:
        window_zscore = window_zscore.union(result)

In [21]:
window_zscore.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- symbol: string (nullable = true)
 |-- window_type: string (nullable = false)
 |-- zscore_price: double (nullable = true)



In [37]:
join_query = window_zscore.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

join_query.awaitTermination()

25/06/16 13:34:44 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-1bc205e0-59fc-486a-8802-f47b352bbaa8. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/06/16 13:34:44 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+---------+------+-----------+------------+
|timestamp|symbol|window_type|zscore_price|
+---------+------+-----------+------------+
+---------+------+-----------+------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-------------------+-------+-----------+--------------------+
|          timestamp| symbol|window_type|        zscore_price|
+-------------------+-------+-----------+--------------------+
|2025-06-16 13:35:00|BTCUSDT|        30s|  3.1648970402662684|
|2025-06-16 13:35:01|BTCUSDT|        30s|    1.17230726887712|
|2025-06-16 13:35:02|BTCUSDT|        30s|0.032799831625969146|
|2025-06-16 13:35:04|BTCUSDT|        30s|0.032799831625969146|
|2025-06-16 13:35:05|BTCUSDT|        30s|0.032799831625969146|
|2025-06-16 13:35:06|BTCUSDT|        30s| -0.3798071412363651|
|2025-06-16 13:35:07|BTCUSDT|        30s|  -0.651523928244642|
|2025-06-16 13:35:09|BTCUSDT|        30s|  -0.651523928244642|
|2025-06-16 13:35:10|BTCUSDT|        30s|  0.6590644489747247|
|2025-06-16 13:35:11|BTCUSDT|        30s|  0.6582903270736974|
|2025-06-16 13:35:12|BTCUSDT|        30s|-0.24123932108201035|
|2025-06-16 13:35:13|

ERROR:root:KeyboardInterrupt while sending command.            (166 + 4) / 1200]
Traceback (most recent call last):
  File "/opt/bitnami/python/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/python/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/python/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

[Stage 186:=========>                                          (217 + 4) / 1200]

In [39]:
join_query.stop()

Group windows z-score to a single JSON record by timestamp and symbol

In [22]:
# Group windows z-score to a single JSON record by timestamp and symbol
grouped = window_zscore.groupBy("symbol", "timestamp").agg(
    collect_list(
        struct(
            col("window_type").alias("window"), 
            col("zscore_price")
        )
    ).alias("windows")
).select(
    to_json(
        struct(
            col("timestamp").cast("string"),
            col("symbol"),
            col("windows")
        )
    ).alias("value")
)

Write into Kafka topic `btc-price-zscore`

In [ ]:
# Write into Kafka topic btc-price-zscore
grouped.writeStream.format("kafka") \
    .option("kafka.bootstrap.servers", "broker:9092") \
    .option("topic", "btc-price-zscore").option("checkpointLocation", '/tmp/btc-zscore-checkpoint') \
    .outputMode("update") \
    .trigger(processingTime="30 seconds") \
    .start().awaitTermination()